In [1]:
# --- 必要なパッケージ ---
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)
library(earth)

# --- 設定 ---
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_degree", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")

result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# --- メインループ ---
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath, row.names = 1)
  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n--- Training for:", target_var, " ---\n")

    # --- 前処理: 欠損除去 + 数値列限定 + 定数列除去 --- #
    df <- df_all[, c(feature_vars, target_var)]
    df <- df[complete.cases(df), ]
    df <- df[, sapply(df, is.numeric)]  # 数値列のみに限定
    nzv <- nearZeroVar(df, saveMetrics = TRUE)
    df <- df[, !(nzv$zeroVar | nzv$nzv)]  # 定数列・ゼロ分散除去

    if (any(!is.finite(as.matrix(df)))) {
      cat("\u26d4\ufe0f 非有限値 (NA/NaN/Inf) が残っています: ", file, target_var, "\n")
      next
    }

    ctrl <- trainControl(method = "cv", number = 10)
    tune_grid <- expand.grid(degree = 1:3)

    model <- train(
      formula(paste(target_var, "~ .")),
      data = df,
      method = "gcvEarth",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid
    )

    pred <- predict(model, df)
    obs <- df[[target_var]]

    R2 <- round(cor(pred, obs)^2, 3)
    RMSE_val <- round(rmse(obs, pred), 3)
    MAE_val <- round(mae(obs, pred), 3)
    RPD_val <- round(sd(obs) / RMSE_val, 3)

    result_matrix[paste0("Best_degree_", target_var), file] <- model$bestTune$degree
    result_matrix[paste0("R2_", target_var), file] <- R2
    result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file] <- MAE_val
    result_matrix[paste0("RPD_", target_var), file] <- RPD_val
    result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
    result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

    # --- 重要度の保存 ---
    vi <- varImp(model)$importance
    vi$Variable <- rownames(vi)
    vi <- vi[order(-vi$Overall), ]
    vi_file <- paste0("new20250702_varimp_gcvEarth_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".csv")
    write.csv(vi, vi_file, row.names = FALSE)

    # --- モデル保存 ---
    model_data_bundle <- list(model = model, training_data = df)
    rds_file <- paste0("new20250702_model_data_gcvEarth_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
    saveRDS(model_data_bundle, file = rds_file)

    # --- プロット関数 ---
    get_axis_limit <- function(values) {
      max_val <- max(values, na.rm = TRUE)
      if (max_val <= 1.5) return(ceiling(max_val / 0.1) * 0.1)
      else if (max_val <= 5) return(ceiling(max_val / 0.5) * 0.5)
      else if (max_val <= 30) return(ceiling(max_val / 2) * 2)
      else return(ceiling(max_val / 5) * 5)
    }

    range_max <- get_axis_limit(c(obs, pred))
    p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
      geom_point(color = "blue", alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_x_continuous(limits = c(0, range_max)) +
      scale_y_continuous(limits = c(0, range_max)) +
      coord_fixed(ratio = 1) +
      labs(
        title = paste0(target_var, " Prediction (", file, ")"),
        x = "Observed", y = "Predicted"
      ) +
      annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
               label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
      annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
               label = paste0("R\u00b2 = ", R2)) +
      theme_minimal()

    pdf_file <- paste0("new20250702_plot_gcvEarth_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".pdf")
    pdf(pdf_file, width = 6, height = 6)
    print(p)
    dev.off()
  }
}

# --- 評価指標の保存 ---
output_file <- paste0("new20250702_gcvEarth_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\n\u2705 Summary saved as:", output_file, "\n")


Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'Metrics'


The following objects are masked from 'package:caret':

    precision, recall


Loading required package: Formula

Loading required package: plotmo

Loading required package: plotrix




=== Processing: n_base.csv ===

--- Training for: Jsc  ---


Warning message:
"Removed 4 rows containing missing values or values outside the scale range
(`geom_point()`)."



--- Training for: Voc  ---

--- Training for: FF  ---

--- Training for: PCEmax  ---


Warning message:
"Removed 18 rows containing missing values or values outside the scale range
(`geom_point()`)."


...


=== Processing: m_all_FP.csv ===

--- Training for: Jsc  ---


Warning message:
"Removed 7 rows containing missing values or values outside the scale range
(`geom_point()`)."



--- Training for: Voc  ---

--- Training for: FF  ---

--- Training for: PCEmax  ---


Warning message:
"Removed 17 rows containing missing values or values outside the scale range
(`geom_point()`)."



<U+2705> Summary saved as: new20250702_gcvEarth_summary_20250702.csv 


In [5]:
# --- 必要なパッケージ ---
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)
library(earth)

# --- 設定 ---
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_degree", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")

result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# --- メインループ ---
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath, row.names = 1)
  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n--- Training for:", target_var, " ---\n")

        # --- 前処理: NA列除去 → 数値列限定 → 定数列除去 → サンプル欠損除去 --- #
    df <- df_all[, c(feature_vars, target_var)]

    # ① NAを多く含む列を除去（例：80%以上欠損）
    na_ratio <- colMeans(is.na(df))
    df <- df[, na_ratio < 0.8]

    # ② 数値列に限定
    df <- df[, sapply(df, is.numeric)]

    # ③ 定数列（SD=0）や near-zero variance の列を除去
    nzv <- nearZeroVar(df, saveMetrics = TRUE)
    df <- df[, !(nzv$zeroVar | nzv$nzv)]

    # ④ 欠損を含むサンプルを除去
    df <- df[complete.cases(df), ]

    # ⑤ 非有限値チェック
    if (any(!is.finite(as.matrix(df)))) {
      cat("\u26d4\ufe0f 非有限値 (NA/NaN/Inf) が残っています: ", file, target_var, "\n")
      next
    }
    cat("Data size:", nrow(df), "rows ×", ncol(df), "columns\n")
    }
}


#     if (any(!is.finite(as.matrix(df)))) {
#       cat("\u26d4\ufe0f 非有限値 (NA/NaN/Inf) が残っています: ", file, target_var, "\n")
#       next
#     }

#     ctrl <- trainControl(method = "cv", number = 10)
#     tune_grid <- expand.grid(degree = 1:3)

#     model <- train(
#       formula(paste(target_var, "~ .")),
#       data = df,
#       method = "gcvEarth",
#       metric = "RMSE",
#       trControl = ctrl,
#       tuneGrid = tune_grid
#     )

#     pred <- predict(model, df)
#     obs <- df[[target_var]]

#     R2 <- round(cor(pred, obs)^2, 3)
#     RMSE_val <- round(rmse(obs, pred), 3)
#     MAE_val <- round(mae(obs, pred), 3)
#     RPD_val <- round(sd(obs) / RMSE_val, 3)

#     result_matrix[paste0("Best_degree_", target_var), file] <- model$bestTune$degree
#     result_matrix[paste0("R2_", target_var), file] <- R2
#     result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
#     result_matrix[paste0("MAE_", target_var), file] <- MAE_val
#     result_matrix[paste0("RPD_", target_var), file] <- RPD_val
#     result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
#     result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

#     # --- 重要度の保存 ---
#     vi <- varImp(model)$importance
#     vi$Variable <- rownames(vi)
#     vi <- vi[order(-vi$Overall), ]
#     vi_file <- paste0("new20250702_varimp_gcvEarth_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".csv")
#     write.csv(vi, vi_file, row.names = FALSE)

#     # --- モデル保存 ---
#     model_data_bundle <- list(model = model, training_data = df)
#     rds_file <- paste0("new20250702_model_data_gcvEarth_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
#     saveRDS(model_data_bundle, file = rds_file)

#     # --- プロット関数 ---
#     get_axis_limit <- function(values) {
#       max_val <- max(values, na.rm = TRUE)
#       if (max_val <= 1.5) return(ceiling(max_val / 0.1) * 0.1)
#       else if (max_val <= 5) return(ceiling(max_val / 0.5) * 0.5)
#       else if (max_val <= 30) return(ceiling(max_val / 2) * 2)
#       else return(ceiling(max_val / 5) * 5)
#     }

#     range_max <- get_axis_limit(c(obs, pred))
#     p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
#       geom_point(color = "blue", alpha = 0.7) +
#       geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
#       scale_x_continuous(limits = c(0, range_max)) +
#       scale_y_continuous(limits = c(0, range_max)) +
#       coord_fixed(ratio = 1) +
#       labs(
#         title = paste0(target_var, " Prediction (", file, ")"),
#         x = "Observed", y = "Predicted"
#       ) +
#       annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
#                label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
#       annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
#                label = paste0("R\u00b2 = ", R2)) +
#       theme_minimal()

#     pdf_file <- paste0("new20250702_plot_gcvEarth_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".pdf")
#     pdf(pdf_file, width = 6, height = 6)
#     print(p)
#     dev.off()
#   }
# }

# # --- 評価指標の保存 ---
# output_file <- paste0("new20250702_gcvEarth_summary_", today, ".csv")
# write.csv(result_matrix, output_file, row.names = TRUE)
# cat("\n\u2705 Summary saved as:", output_file, "\n")


Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'Metrics'


The following objects are masked from 'package:caret':

    precision, recall


Loading required package: Formula

Loading required package: plotmo

Loading required package: plotrix




=== Processing: n_base.csv ===

--- Training for: Jsc  ---
Data size: 358 rows <U+00D7> 8 columns

--- Training for: Voc  ---
Data size: 358 rows <U+00D7> 8 columns

--- Training for: FF  ---
Data size: 358 rows <U+00D7> 8 columns

--- Training for: PCEmax  ---
Data size: 358 rows <U+00D7> 8 columns

=== Processing: n_base_OH.csv ===

--- Training for: Jsc  ---
Data size: 358 rows <U+00D7> 11 columns

--- Training for: Voc  ---
Data size: 358 rows <U+00D7> 11 columns

--- Training for: FF  ---
Data size: 358 rows <U+00D7> 11 columns

--- Training for: PCEmax  ---
Data size: 358 rows <U+00D7> 11 columns

=== Processing: n_base_FP.csv ===

--- Training for: Jsc  ---
Data size: 358 rows <U+00D7> 79 columns

--- Training for: Voc  ---
Data size: 358 rows <U+00D7> 79 columns

--- Training for: FF  ---
Data size: 358 rows <U+00D7> 79 columns

--- Training for: PCEmax  ---
Data size: 358 rows <U+00D7> 79 columns

=== Processing: n_all.csv ===

--- Training for: Jsc  ---
Data size: 72 rows <U